<a href="https://colab.research.google.com/github/gonzalo-agostino/detector-parking-yolo/blob/main/ParkingDetectionModel_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Oct 22 12:30:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [20]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install --upgrade pip
!pip -q install ultralytics opencv-python matplotlib
!yolo checks


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.3.219 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 44.4/235.7 GB disk)

OS                     Linux-6.6.105+-x86_64-with-glibc2.35
Environment            Colab
Python                 3.12.12
Install                pip
Path                   /usr/local/lib/python3.12/dist-packages/ultralytics
RAM                    167.05 GB
Disk                   44.4/235.7 GB
CPU                    Intel Xeon CPU @ 2.20GHz
CPU count              12
GPU                    NVIDIA A100-SXM4-80GB, 81222MiB
GPU count              1
CUDA                   12.6

numpy                  ✅ 2.0.2>=1.23.0
matplotlib             ✅ 3.10.0>=3.3.0
opencv-python          ✅ 4.12.0.88>=4.6.0
pillow                 ✅ 11.3.0>=7.1.2
pyyaml                 ✅ 6.0.3>=5.3.1
requests               ✅ 2.32.4>=2.23.0
s

In [ ]:
DATASET_DIR  = "/content/drive/MyDrive/ParkingDetection/dataset"
DATASET_YAML = "/content/drive/MyDrive/ParkingDetection/dataset.yaml"

# Vista rápida de la estructura
!ls -R "$DATASET_DIR" | head -n 200


/content/drive/MyDrive/ParkingDetection/dataset:
dataset.yaml
test
train
valid

/content/drive/MyDrive/ParkingDetection/dataset/test:
images
labels

/content/drive/MyDrive/ParkingDetection/dataset/test/images:
002_jpg.rf.2a50555873cb5aac7a286818be9d2404.jpg
003_jpg.rf.d05b2d5a38bfa73f8c8cec4b599e0e7d.jpg
006_jpg.rf.99541524a9fa02468ee4b95231f55dfd.jpg
017_jpg.rf.eadb9b856ad2c3f7e719269a9b6f5cbe.jpg
022_jpg.rf.e0decd92836a3fdd6b8b23f0a0ecae9d.jpg
024_jpg.rf.65861fa1e82a748d368157ae8c4d6528.jpg
043_jpg.rf.2bf0f2d3239ccefeaa52993d7eedb198.jpg
047_jpg.rf.2ef1ba2580de8d3a2e40faf34c550ab6.jpg
077_jpg.rf.8b70795853e5b82619025afac215a0ca.jpg
087_jpg.rf.21eceb818379d6b8a6c72919c6cb698d.jpg
098_jpg.rf.7b05aba59d602d1a77a62476710e39ea.jpg
113_jpg.rf.dbd01cca622cc80ccb46c0e0b1c83bae.jpg
114_jpg.rf.a01facfa9c450b4649b1cffd37285625.jpg
117_jpg.rf.2158cd0a38d471208ddafd179df82e26.jpg
123_jpg.rf.799645afc97db3b0facff9e8a5fbcea6.jpg
140_jpg.rf.cda07c7803b2ad70d71c2fa6e6eed730.jpg
169_jpg.rf.17ded3d71d5

In [ ]:
import glob, os
base = DATASET_DIR
for split in ["train","valid","test"]:
    n_img = len(glob.glob(os.path.join(base, "images", split, "*.*")))
    n_lbl = len(glob.glob(os.path.join(base, "labels", split, "*.txt")))
    print(f"{split:5s}  imgs: {n_img:5d}   labels: {n_lbl:5d}")

train  imgs:     0   labels:     0
valid  imgs:     0   labels:     0
test   imgs:     0   labels:     0


In [ ]:
import os, glob, yaml

DATASET_DIR  = "/content/drive/MyDrive/ParkingDetection/dataset"
DATASET_YAML = f"{DATASET_DIR}/dataset.yaml"

yaml_text = f"""
path: {DATASET_DIR}
train: train/images
val: valid/images
test: test/images
names:
  0: car
"""
with open(DATASET_YAML, "w", encoding="utf-8") as f:
    f.write(yaml_text)

print("dataset.yaml actualizado:")
print(yaml_text)

# Recontar
def count_split(root, rel):
    img_dir = os.path.join(root, rel)
    lbl_dir = img_dir.replace("/images", "/labels")
    exts = ("*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG")
    n_img = sum(len(glob.glob(os.path.join(img_dir, e))) for e in exts)
    n_lbl = len(glob.glob(os.path.join(lbl_dir, "*.txt")))
    return img_dir, lbl_dir, n_img, n_lbl

with open(DATASET_YAML, "r", encoding="utf-8") as f:
    y = yaml.safe_load(f)

for key in ["train","val","test"]:
    rel = y.get(key)
    if not rel: continue
    img_dir, lbl_dir, n_img, n_lbl = count_split(y["path"], rel)
    print(f"{key:5s} imgs: {n_img:5d}  labels: {n_lbl:5d}   ({img_dir})")


dataset.yaml actualizado:

path: /content/drive/MyDrive/ParkingDetection/dataset
train: train/images
val: valid/images
test: test/images
names:
  0: car

train imgs: 11586  labels: 11586   (/content/drive/MyDrive/ParkingDetection/dataset/train/images)
val   imgs:   794  labels:   794   (/content/drive/MyDrive/ParkingDetection/dataset/valid/images)
test  imgs:    18  labels:    18   (/content/drive/MyDrive/ParkingDetection/dataset/test/images)


In [ ]:
print("Copiando dataset al disco local rápido de la A100...")
!cp -r /content/drive/MyDrive/ParkingDetection/dataset /content/dataset
print("¡Copia terminada! El dataset está en /content/dataset")

Copiando dataset al disco local rápido de la A100...
¡Copia terminada! El dataset está en /content/dataset


In [ ]:
# ¡CAMBIO CLAVE! Apuntamos al YAML que acabamos de copiar al disco local
DATASET_YAML = "/content/dataset/dataset.yaml"

# Dónde guardar los resultados (en tu Drive)
SAVE_DIR = "/content/drive/MyDrive/ParkingDetection/YOLO_Training_Runs"


# ---------------------------------------------------------------
#  Ejecutar el entrenamiento NUEVO
# ---------------------------------------------------------------
# - model='yolo11s.pt' -> Empieza desde el modelo original.
# - (SIN resume=True) -> Comienza desde la Epoch 1.
# - name='run_A100_from_scratch' -> Guarda en una carpeta nueva.
# - optimizer='auto' -> Permite que la A100 use el mejor optimizador (AdamW).

!yolo detect train \
  model='yolo11s.pt' \
  data="$DATASET_YAML" \
  project="$SAVE_DIR" \
  name='run_A100' \
  imgsz=640 \
  epochs=100 \
  batch=-1 \
  optimizer='auto' \
  cos_lr=True \
  warmup_epochs=3 \
  patience=30 \
  amp=True \
  mosaic=0.7 \
  hsv_h=0.015 hsv_s=0.7 hsv_v=0.4 \
  degrees=5 translate=0.05 scale=0.1 shear=2 perspective=0.0005 \
  fliplr=0.2 flipud=0.0 \
  cache=True

Ultralytics 8.3.219 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/dataset.yaml, degrees=5, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.2, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.7, multi_scale=False, name=run_A100, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, perspective=0.0005, pl

In [25]:
# 1. Ruta a tu modelo entrenado
MODELO_ENTRENADO = "/content/drive/MyDrive/ParkingDetection/YOLO_Training_Runs/run_A100/weights/best.pt"

# 2. Ruta al video que quieres probar
VIDEO_FUENTE = "/content/drive/MyDrive/ParkingDetection/videos/4video.mp4"

# 3. Dónde guardar los resultados
SAVE_DIR = "/content/drive/MyDrive/ParkingDetection/"

# - model -> Tu archivo best.pt
# - source -> El video (o imagen, o carpeta) que quieres analizar
# - conf=0.5 -> (OPCIONAL) Solo muestra detecciones con más de 50% de confianza.
# - save=True -> Guarda el video con los resultados

!yolo predict \
  model="$MODELO_ENTRENADO" \
  source="$VIDEO_FUENTE" \
  project="$SAVE_DIR" \
  name="test_videos" \
  conf=0.5 \
  save=True \
  exist_ok=True

Ultralytics 8.3.219 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 7, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 990, in entrypoint
    getattr(model, mode)(**overrides)  # default args from model
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 557, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/predictor.py", line 249, in predict_cli
    for _ in gen:  # sourcery skip: remove-empty-nested-block, noqa
             ^^^
  File "/usr/local/